# Open-model tour: five Hugging Face models on a Colab T4

The same prompt through four instruct-tuned models and one reasoning model, at the
tokenizer / `AutoModelForCausalLM` level rather than behind a `pipeline`, so the differences
in chat templating and output quality are visible directly.

| Model | Params | Gated |
| --- | --- | --- |
| `meta-llama/Llama-3.2-1B-Instruct` | 1.2B | **yes** |
| `microsoft/Phi-4-mini-instruct` | 3.8B | no |
| `google/gemma-3-270m-it` | 0.27B | **yes** |
| `Qwen/Qwen3-4B-Instruct-2507` | 4B | no |
| `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` | 1.5B | no |

Every model is loaded in 4-bit and freed again before the next one, so the 16 GB of a T4 is
never close to full and the model cells can be run in any order, as often as you like.

## Before you run anything

**1. Pick a GPU runtime.** *Runtime → Change runtime type → T4 GPU*. On a CPU runtime the
first model load fails inside `bitsandbytes` with an error that does not explain itself.

**2. Add your Hugging Face token.** Click the 🔑 key icon in the left sidebar, add a secret
named `HF_TOKEN` holding a token from
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens), and switch on
*Notebook access*. Never paste the token into a cell — it would be saved into the notebook.

Colab's secret store only answers requests made **from the Colab UI in a browser tab**, so if
you are driving this runtime from somewhere else — a local IDE, an automated run — the token
cell will report `TimeoutException` and then prompt you for the token instead. That prompt is
the intended fallback, not a failure; what you type is not saved into the notebook.

**3. Accept the licences for the two gated models**, using the same account the token belongs
to. Access is granted per model, so approval for Llama 3.1 does *not* carry over to 3.2:

- [meta-llama/Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct)
- [google/gemma-3-270m-it](https://huggingface.co/google/gemma-3-270m-it)

Approval usually lands within a few minutes. Without it, the Llama and Gemma cells raise a
`GatedRepoError` — the other three still work, token or no token.

In [ ]:
# Colab preinstalls transformers, but the pinned version drifts and gemma-3 (>=4.50) and
# Qwen3 (>=4.51) both need recent releases. bitsandbytes is not preinstalled at all.
!pip install -q -U transformers accelerate bitsandbytes

In [4]:
import gc
import os
from enum import StrEnum
from getpass import getpass

import torch
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TextStreamer,
)


class Model(StrEnum):
    """The five models on the tour. StrEnum, so members pass straight to from_pretrained."""

    LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
    PHI = "microsoft/Phi-4-mini-instruct"
    GEMMA = "google/gemma-3-270m-it"
    QWEN = "Qwen/Qwen3-4B-Instruct-2507"
    DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"


# One quantization config for all five, so the comparison is apples-to-apples. nf4 with double
# quantization puts even Qwen3-4B (~8 GB in fp16) at roughly 2.5 GB.
QUANT = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    # float16, NOT bfloat16: the T4 is a Turing card and has no bf16 support.
    bnb_4bit_compute_dtype=torch.float16,
)

In [5]:
# Three ways to find a token, in order of preference, because Colab's secret store is only
# reachable from the Colab UI and is not always reachable even there.
token = os.environ.get("HF_TOKEN")

try:
    from google.colab import userdata
except ImportError:
    pass  # Not running on Colab — the environment variable above is the whole story.
else:
    try:
        token = userdata.get("HF_TOKEN") or token
    except Exception as exc:
        # userdata.get raises rather than returning None: TimeoutException when the request
        # cannot reach the Colab UI, SecretNotFoundError when the secret does not exist, and
        # NotebookAccessError when it exists but this notebook is not allowed to read it.
        print(f"Colab secret HF_TOKEN unavailable ({type(exc).__name__}).")

if not token:
    # Last resort. getpass keeps the token off the screen and out of the saved notebook —
    # which is the one thing you must not do, since outputs and source are both saved.
    token = getpass("HF token (press Enter to skip the gated models): ").strip()

if token:
    login(token=token)
    print("Logged in to Hugging Face.")
else:
    print("No token — Phi, Qwen and DeepSeek will still run; Llama and Gemma will not.")

Colab secret HF_TOKEN unavailable (TimeoutException).
Logged in to Hugging Face.


In [6]:
def generate(
    model: Model, messages: list[dict[str, str]], max_new_tokens: int = 300
) -> str:
    """Load `model` in 4-bit, stream its reply to `messages`, and give the GPU back."""
    tokenizer = AutoTokenizer.from_pretrained(model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Each family templates a conversation its own way — that is part of what we are comparing.
    # add_generation_prompt appends the "the assistant speaks now" marker the model expects.
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")

    llm = AutoModelForCausalLM.from_pretrained(
        model, quantization_config=QUANT, device_map="auto"
    )
    try:
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            # skip_prompt so the live stream shows the reply only, not the templated prompt.
            streamer=TextStreamer(tokenizer, skip_prompt=True),
        )
        # generate() returns prompt + reply; slice the prompt off so the return value is usable.
        new_tokens = outputs[0][inputs["input_ids"].shape[-1] :]
        return tokenizer.decode(new_tokens, skip_special_tokens=True)
    finally:
        # Load-bearing, not ceremony: this is what lets the cells below be run in any order and
        # as often as you like without VRAM piling up — including after generate() raises.
        del llm
        gc.collect()
        torch.cuda.empty_cache()

In [9]:
messages = [{"role": "user", "content": "Tell a joke for a room of guitarists"}]

## The tour

One cell per model, so any of them can be rerun on its own. The reply streams out as it
is generated; it is also bound to `reply` if you want to pick it apart afterwards.

In [10]:
reply = generate(Model.LLAMA, messages)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Why did the guitar go to therapy?

Because it was feeling a little "out of tune" and wanted to work through some "harmony" issues.<|eot_id|>


In [ ]:
reply = generate(Model.PHI, messages)

config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
reply = generate(Model.GEMMA, messages)

In [ ]:
reply = generate(Model.QWEN, messages)

In [ ]:
# 800 rather than 300: DeepSeek thinks out loud in a <think> block before answering,
# and at 300 tokens it never gets to the joke.
reply = generate(Model.DEEPSEEK, messages, max_new_tokens=800)

## What to notice

**Size shows.** Gemma 3 270M has roughly a fifteenth of Qwen3-4B's parameters, and the joke is
where that gap becomes obvious: the small model tends to produce something joke-shaped rather
than something funny. Reruns vary — sampling is on by default — so run a cell a few times
before drawing conclusions about any one model.

**DeepSeek is doing something different.** It is a reasoning distill, not an instruct model.
Its chat template ends by opening a `<think>` block for the model, which then reasons in the
open — often for hundreds of tokens — closes with `</think>`, and only then tells the joke.
That is why its cell gets `max_new_tokens=800`: at 300 it gets cut off mid-thought and never
reaches the punchline. Compare its raw output with the other four and the split between
thinking and answering is plain.

**Templates are not interchangeable.** Print one to see what the model actually received:

```python
tokenizer = AutoTokenizer.from_pretrained(Model.QWEN)
print(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))
```

Qwen uses `<|im_start|>user … <|im_end|>`, Phi uses `<|user|> … <|end|>`, DeepSeek uses
`<｜User｜> … <｜Assistant｜><think>`. Hand-rolling these strings instead of calling
`apply_chat_template` is a reliable way to get quietly degraded output.

**Things worth trying next:** swap in a system turn and watch which families accept a `system`
role and which fold it into the user message; drop `QUANT` for a model small enough to run in
fp16 and compare quality; or set `do_sample=False` for reproducible output across runs.